# 09 · NEON coverage, verified image set, predefined features and temporal dependence

**Scope.** HARV and CPER, June–August 2023, exploratory (review 3). This notebook builds the analysis table that notebook 10 uses:
1. labels with the measurement model of notebook 08;
2. every hourly frame, with its clock and camera configuration verified from its `.meta` file;
3. the hourly JPEG set, with SHA-256 manifest;
4. nine predefined, overlay-masked scene features (`features.py`, decision D-021).

It then measures temporal dependence in the label, the context residual and the visual residuals. Visual features are **not** related to the label here.

Only hourly frames (on the hour, 06–19 local standard time) and their `.meta` files were downloaded, not every half-hourly frame. This kept the PhenoCam archive load and disk use moderate (1.22 GB).

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from visual_hygrometry import data, features, inference, neon, scoring, splits, thermo, worlds

plt.rcParams.update({'figure.dpi': 80, 'figure.figsize': (11, 3.2), 'axes.grid': True, 'grid.alpha': .3})
pd.set_option('display.precision', 3)
MONTHS = ['2023-06', '2023-07', '2023-08']
SITES = {'HARV': ('NEON.D01.HARV.DP1.00033', -5), 'CPER': ('NEON.D10.CPER.DP1.00033', -7)}


def labels(site):
    """Notebook 08 measurement model: D_h = T_asp - Td_probe leading, D_probe sensitivity, inconsistent minutes (D_h < -2 sd) excluded."""
    s = neon.load_site(site, MONTHS)
    s['D_h'], s['D_probe'] = s.t_asp - s.td, s.t_probe - s.td
    s['sd_Dh'] = np.hypot(s.td_unc / 2, s.t_asp_unc / 2)
    s['rh_implied'] = thermo.rh_from_dewpoint(s.t_probe, s.td)
    base = (s[['td_qf', 't_asp_qf', 't_probe_qf']] == 0).all(axis=1) & s[['td', 't_asp', 't_probe']].notna().all(axis=1)
    s['inconsistent'] = base & (s.D_h < -2 * s.sd_Dh)
    s['valid'] = base & ~s.inconsistent
    return s

L = {site: labels(site) for site in SITES}
pd.DataFrame([dict(site=site, month=m, minutes=len(g), valid_label=g.valid.mean(), inconsistent_minutes=int(g.inconsistent.sum()), sw_present=g.sw.notna().mean())
              for site, s in L.items() for m, g in s.groupby(s.index.strftime('%Y-%m'))]).set_index(['site', 'month'])

minutes  valid_label  inconsistent_minutes  sw_present
site month                                                          
HARV 2023-06    43200        0.974                     9       0.975
     2023-07    44640        0.995                    24       0.997
     2023-08    44640        0.997                     5       0.998
CPER 2023-06    43200        0.986                   300       0.998
     2023-07    44640        0.982                   659       0.998
     2023-08    44640        0.998                     0       0.998

## 1 · Frames: listing, clock and camera configuration

In [2]:
F = {site: neon.load_frames(cam, MONTHS, off) for site, (cam, off) in SITES.items()}
hourly = lambda f: (f.base.str[-4:-2] == '00') & f.base.str[-6:-4].astype(int).between(6, 19)  # on the hour, 06-19 LST
rows = []
for site, f in F.items():
    for m, g in f.groupby('month'):
        h = g[hourly(g)]
        rows.append(dict(site=site, month=m, frames_listed=len(g), hourly_listed=len(h), hourly_meta=h.overlay_utc.notna().mean(), hourly_jpg_on_disk=int(h.has_jpg.sum()),
                         max_clock_offset_s=(h.overlay_utc - h.utc).dt.total_seconds().abs().max(),
                         **{f'distinct_{k}': h[k].nunique() for k in ['overlay_pos', 'auto_haze', 'agc', 'auto_exposure', 'wb', 'resolution', 'ir_enable']}))
cfg = pd.DataFrame(rows).set_index(['site', 'month'])
cfg

frames_listed  hourly_listed  hourly_meta  hourly_jpg_on_disk  \
site month                                                                    
HARV 2023-06           1078            419        1.000                 419   
     2023-07           1082            433        1.000                 433   
     2023-08            998            434        1.000                 434   
CPER 2023-06           1050            420        0.998                 420   
     2023-07           1066            434        1.000                 434   
     2023-08            989            434        1.000                 434   

              max_clock_offset_s  distinct_overlay_pos  distinct_auto_haze  \
site month                                                                   
HARV 2023-06                 0.0                     1                   1   
     2023-07                 0.0                     1                   1   
     2023-08                 0.0                     1                   1   
CPER 2023-06                 0.0                     1                   1   
     2023-07                 0.0                     1                   1   
     2023-08                 0.0                     1                   1   

              distinct_agc  distinct_auto_exposure  distinct_wb  \
site month                                                        
HARV 2023-06             1                       1            1   
     2023-07             1                       1            1   
     2023-08             1                       1            1   
CPER 2023-06             1                       1            1   
     2023-07             1                       1            1   
     2023-08             1                       1            1   

              distinct_resolution  distinct_ir_enable  
site month                                             
HARV 2023-06                    1                   1  
     2023-07                    1                   1  
     2023-08                    1                   1  
CPER 2023-06                    1                   1  
     2023-07                    1                   1  
     2023-08                    1                   1

In [3]:
print({site: {k: f[hourly(f)][k].dropna().unique().tolist() for k in ['overlay_pos', 'auto_haze', 'agc', 'wb', 'resolution', 'ir_enable']} for site, f in F.items()})

{'HARV': {'overlay_pos': ['0/0/0'], 'auto_haze': ['1'], 'agc': ['1'], 'wb': ['385/256/330'], 'resolution': ['6'], 'ir_enable': ['0']}, 'CPER': {'overlay_pos': ['0/0/0'], 'auto_haze': ['1'], 'agc': ['1'], 'wb': ['385/256/330'], 'resolution': ['6'], 'ir_enable': ['0']}}


## 2 · Image set and manifest

Every hourly JPEG on disk is hashed. The manifest (names, UTC times, sizes, SHA-256) goes to `data/manifests/`; the images stay in `data/raw/` (ignored by Git; PhenoCam archive, CC BY 4.0).

In [4]:
man = pd.concat([f[f.has_jpg & hourly(f)].assign(site=site) for site, f in F.items()])
man['sha256'] = Parallel(n_jobs=8)(delayed(neon.sha256)(p) for p in man.jpg)
man['bytes'] = man.jpg.map(lambda p: p.stat().st_size)
out = data.ROOT / 'data' / 'manifests'
out.mkdir(exist_ok=True)
man[['site', 'base', 'utc', 'bytes', 'sha256']].to_csv(out / 'phenocam_hourly_2023JJA.csv', index=False)
print(f'{len(man)} hourly frames hashed ({man.bytes.sum() / 1e9:.2f} GB); duplicate hashes: {man.sha256.duplicated().sum()}')

2574 hourly frames hashed (1.22 GB); duplicate hashes: 0


## 3 · Join, usable rows and features

A frame is usable when its JPEG and `.meta` are on disk (so its clock and camera state are verified), the 1-min record containing its verified UTC time carries a valid label (§0), and incoming shortwave exceeds 10 W m⁻² (daylight). Features are extracted only for usable frames.

In [5]:
def site_table(site):
    f = F[site]
    h = f[f.has_jpg & hourly(f) & f.overlay_utc.notna()].copy()
    h['minute'] = h.utc.dt.floor('1min')
    j = h.join(L[site], on='minute')
    counts = dict(site=site, hourly_jpg=int((f.has_jpg & hourly(f)).sum()), with_meta=len(h), with_label_record=int(j.td.notna().sum()), valid_label=int(j.valid.fillna(False).sum()),
                  valid_and_daylit=int((j.valid.fillna(False) & (j.sw > 10)).sum()))
    j = j[j.valid.fillna(False) & (j.sw > 10)].reset_index(drop=True)
    j = j.join(pd.DataFrame(Parallel(n_jobs=12)(delayed(features.frame_features)(p, site) for p in j.jpg)))
    j['day'] = (j.utc + pd.Timedelta(hours=SITES[site][1])).dt.tz_localize(None).dt.floor('D')
    counts |= dict(usable_rows=len(j), days=j.day.nunique(), days_with_6_rows=int((j.groupby('day').size() >= 6).sum()))
    return j, counts

T, C = {}, []
for site in SITES:
    T[site], c = site_table(site)
    C.append(c)
interim = data.ROOT / 'data' / 'interim'
interim.mkdir(exist_ok=True)
for site, t in T.items(): t.drop(columns=['jpg']).to_csv(interim / f'm1_{site}.csv', index=False)
pd.DataFrame(C).set_index('site')

,hourly_jpg,with_meta,with_label_record,valid_label,valid_and_daylit,usable_rows,days,days_with_6_rows
site,,,,,,,,
HARV,1286,1286,1273,1271,1197,1197,92,92
CPER,1288,1287,1283,1263,1206,1206,92,92


## 4 · Feature quality

**Declared rule** (image QC only, set before any label–feature relation is examined): a site's two sky features are dropped if the median share of clipped sky pixels exceeds 0.2.

In [6]:
q = pd.DataFrame({site: {'median sky clip share': t.qc_sky_clip.median(), 'frames with > 50 % sky clipped': (t.qc_sky_clip > .5).mean(),
                         'drop sky features (rule)': t.qc_sky_clip.median() > .2} for site, t in T.items()}).T
display(q)
display(pd.concat({site: t[features.NAMES].describe(percentiles=[.05, .5, .95]).T[['mean', '5%', '50%', '95%']] for site, t in T.items()}, axis=1))
conf = pd.concat({site: t[features.NAMES + ['exposure', 'sw']].corr().loc[features.NAMES, ['exposure', 'sw']] for site, t in T.items()}, axis=1)
print('correlation of each feature with acquisition and illumination state (no label involved):')
conf

,median sky clip share,frames with > 50 % sky clipped,drop sky features (rule)
HARV,1.0,0.993,True
CPER,0.038,0.364,False


HARV                        CPER                     
                     mean     5%    50%    95%   mean     5%    50%    95%
vis_sky_lum         0.998  0.996  1.000  1.000  0.829  0.529  0.862  1.000
vis_sky_br          1.001  1.000  1.000  1.003  1.130  1.000  1.066  1.428
vis_far_contrast    0.176  0.113  0.172  0.257  0.179  0.123  0.174  0.248
vis_near_contrast   0.549  0.444  0.513  0.724  0.366  0.292  0.353  0.486
vis_contrast_ratio  0.318  0.231  0.321  0.408  0.498  0.332  0.481  0.724
vis_far_sat         0.233  0.079  0.235  0.344  0.349  0.190  0.359  0.468
vis_near_gcc        0.467  0.442  0.467  0.490  0.387  0.349  0.381  0.435
vis_near_dark       0.112  0.063  0.117  0.151  0.187  0.114  0.191  0.243
vis_far_sharp       0.036  0.005  0.031  0.077  0.008  0.003  0.007  0.016

correlation of each feature with acquisition and illumination state (no label involved):


HARV            CPER       
                   exposure     sw exposure     sw
vis_sky_lum           0.037 -0.160    0.260 -0.551
vis_sky_br           -0.006  0.140   -0.215  0.518
vis_far_contrast     -0.396  0.521   -0.148  0.057
vis_near_contrast    -0.375  0.454   -0.056 -0.326
vis_contrast_ratio   -0.267  0.370   -0.111  0.239
vis_far_sat          -0.361  0.652   -0.323  0.396
vis_near_gcc         -0.143  0.207    0.101 -0.034
vis_near_dark         0.349 -0.551   -0.009  0.509
vis_far_sharp        -0.385  0.371    0.007 -0.297

**Reading (image set and features).**
- All 2574 hourly frames (HARV 1286, CPER 1288) are on disk, 1.22 GB in total, with no duplicate hashes. One CPER `.meta` file is missing on the server, and that frame is excluded. For every other frame the overlay clock agrees with the file-name time to 0 s.
- The camera configuration is constant at both sites across all three months: auto-haze on, AGC on, auto-exposure, white balance 385/256/330, fixed overlay, IR off. The acquisition chain is therefore stable, but it is automatic throughout. Exposure varies with the scene and must be treated as a confounder, never as a feature (D-042).
- Usable rows: HARV 1197 and CPER 1206. Each site covers 92 days, and every day has at least 6 rows.
- The sky band at HARV is clipped on almost every frame (median clipped share 1.0), so the predeclared rule drops both sky features there. At CPER the median clipped share is 0.04 and the sky features are kept.
- The features track illumination. At HARV, far and near contrast, far saturation and sharpness correlate +0.37 to +0.65 with incoming shortwave and −0.27 to −0.40 with exposure; near-field darkness correlates −0.55 with shortwave. Cloudiness thus reaches the features directly, and it is not part of the context C of E2.

## 5 · Temporal dependence

Persistence is measured on three daily series (daytime means):
- the label `D_h` itself;
- the **context residual** of `D_h`, from out-of-fold predictions of the context-only model of notebook 10 (hour basis, spline in aspirated T, hour × T, day-of-year spline; contiguous 16-day folds);
- each **visual residual** V − E[V | C], from out-of-fold predictions of the same context model and folds.

Lag-k autocorrelations; N_eff ≈ N (1 − ρ₁)/(1 + ρ₁), meaningful only for moderate ρ₁.

The last column repeats lag 1 for the visual residual of the boosted projection `inference.project`, which notebook 10 adds to the context. Its inner folds interleave days, so each held-out day is predicted partly from its neighbours. That residual is a local difference and is negatively autocorrelated by construction. It does not measure the persistence of the image series.

In [7]:
DOY_KNOTS = np.arange(140, 260.1, 15.0)

def dependence(site, t):
    off = SITES[site][1]
    df = t.assign(temp_cond_c=t.t_asp, hour_local=((t.utc.dt.hour + t.utc.dt.minute / 60 + off) % 24), doy=t.day.dt.day_of_year)
    df = pd.concat([df, data.fourier(df.hour_local, 24, 2, 'hour'), data.spline(df.doy, DOY_KNOTS, 'doy_spl')], axis=1)
    df, ctx = worlds.add_context_basis(df, np.arange(-4, 45, 4.0))
    ctx += [c for c in df if c.startswith('doy_spl')]
    g = (df.day - df.day.min()).dt.days.to_numpy()
    sp = splits.kfold(splits.assign_folds(g // 16, 5, 'contiguous'))
    y = df.D_h.to_numpy()
    r = y - scoring.crossfit(df[ctx].to_numpy(), y, sp, g)[0]
    raw = df[['hour_local', 'temp_cond_c', 'doy']].to_numpy()
    V = df[features.NAMES].to_numpy()
    R = np.column_stack([v - scoring.crossfit(df[ctx].to_numpy(), v, sp, g)[0] for v in V.T])
    P = V - inference.project(raw, V, g)
    daily = lambda R: pd.DataFrame({f'{n} residual': R[:, k] for k, n in enumerate(features.NAMES)}).groupby(g).mean()
    series = pd.DataFrame({'D_h': y, 'context residual of D_h': r}).groupby(g).mean().join(daily(R))
    out = pd.DataFrame({f'lag {k}': series.apply(lambda s: s.autocorr(k)) for k in (1, 2, 3, 7)})
    out['N_days'] = len(series)
    out['N_eff (lag 1)'] = len(series) * (1 - out['lag 1']) / (1 + out['lag 1'])
    out['lag 1, boosted-projection residual'] = daily(P).apply(lambda s: s.autocorr(1))
    return out

dep = pd.concat({site: dependence(site, t) for site, t in T.items()})
dep

lag 1  lag 2  lag 3  lag 7  N_days  \
HARV D_h                          0.196 -0.144 -0.085 -0.108      92   
     context residual of D_h      0.496  0.236  0.157 -0.013      92   
     vis_sky_lum residual         0.018  0.155 -0.033 -0.063      92   
     vis_sky_br residual          0.066  0.108 -0.037 -0.060      92   
     vis_far_contrast residual    0.360  0.245  0.097 -0.104      92   
     vis_near_contrast residual   0.507  0.408  0.293  0.053      92   
     vis_contrast_ratio residual  0.480  0.330  0.160 -0.075      92   
     vis_far_sat residual         0.343  0.179  0.018 -0.136      92   
     vis_near_gcc residual        0.304 -0.002  0.030  0.086      92   
     vis_near_dark residual       0.553  0.470  0.348  0.099      92   
     vis_far_sharp residual       0.479  0.314  0.153  0.032      92   
CPER D_h                          0.568  0.199  0.034  0.196      92   
     context residual of D_h      0.954  0.916  0.871  0.616      92   
     vis_sky_lum residual         0.720  0.572  0.528  0.186      92   
     vis_sky_br residual          0.204  0.081 -0.151 -0.076      92   
     vis_far_contrast residual    0.753  0.678  0.628  0.292      92   
     vis_near_contrast residual   0.934  0.884  0.797  0.480      92   
     vis_contrast_ratio residual  0.820  0.732  0.645  0.257      92   
     vis_far_sat residual         0.685  0.560  0.426  0.350      92   
     vis_near_gcc residual        0.983  0.948  0.898  0.616      92   
     vis_near_dark residual       0.913  0.871  0.776  0.402      92   
     vis_far_sharp residual       0.926  0.872  0.794  0.469      92   

                                  N_eff (lag 1)  \
HARV D_h                                 61.801   
     context residual of D_h             31.011   
     vis_sky_lum residual                88.671   
     vis_sky_br residual                 80.604   
     vis_far_contrast residual           43.339   
     vis_near_contrast residual          30.095   
     vis_contrast_ratio residual         32.308   
     vis_far_sat residual                44.984   
     vis_near_gcc residual               49.152   
     vis_near_dark residual              26.508   
     vis_far_sharp residual              32.450   
CPER D_h                                 25.368   
     context residual of D_h              2.182   
     vis_sky_lum residual                15.005   
     vis_sky_br residual                 60.866   
     vis_far_contrast residual           12.947   
     vis_near_contrast residual           3.120   
     vis_contrast_ratio residual          9.121   
     vis_far_sat residual                17.209   
     vis_near_gcc residual                0.803   
     vis_near_dark residual               4.160   
     vis_far_sharp residual               3.552   

                                  lag 1, boosted-projection residual  
HARV D_h                                                         NaN  
     context residual of D_h                                     NaN  
     vis_sky_lum residual                                     -0.456  
     vis_sky_br residual                                      -0.416  
     vis_far_contrast residual                                -0.389  
     vis_near_contrast residual                               -0.421  
     vis_contrast_ratio residual                              -0.364  
     vis_far_sat residual                                     -0.277  
     vis_near_gcc residual                                    -0.344  
     vis_near_dark residual                                   -0.435  
     vis_far_sharp residual                                   -0.342  
CPER D_h                                                         NaN  
     context residual of D_h                                     NaN  
     vis_sky_lum residual                                     -0.347  
     vis_sky_br residual                                      -0.372  
     vis_far_contrast residual                                -0.29

**Reading (temporal dependence).**
- **HARV.** The daily context residual of `D_h` has lag-1 autocorrelation 0.50 and is near zero by lag 7. The visual residuals are similar (lag 1 between 0.30 and 0.55), giving N_eff ≈ 26–49 of 92 days.
- **CPER is strongly persistent.** The context residual of `D_h` has lag-1 autocorrelation 0.95 and lag-7 autocorrelation 0.62. Most visual residuals have lag 1 of 0.68–0.98, the greenness residual reaching 0.98. The summer at CPER therefore holds only a handful of independent periods. The out-of-fold context, which interpolates season across held-out 16-day blocks, leaves a regime-scale anomaly that the image (probably the vegetation state) also carries.
- The boosted-projection residual used by notebook 10 has negative lag-1 autocorrelation (−0.06 to −0.46). As explained above, that projection interpolates between neighbouring days on its own inner folds. This artefact has consequences in notebook 10 §3.
- For the cyclic-shift null these series are acceptable only under stationarity. A monsoon onset and a phenology cycle within one summer make that assumption doubtful at CPER.